# Does Regularization Shape Prunability? — experiment runner
Settings → Accelerator: **GPU T4 x2**, Internet: **On**.
**Add Input → your Kaggle dataset containing `cifar-10-python.tar.gz`** (md5 c58f30108f718f92721af3b95e74349a) to skip the 35-minute download.
Each 100-epoch ResNet-18 run takes ~25 min on a T4; two runs execute in parallel, one per GPU.
Use **Save Version → Save & Run All (Commit)** so outputs in `/kaggle/working` are kept after the session ends.

In [ ]:
import os, subprocess, glob, shutil
REPO = 'https://github.com/WHITELOTUS0/dropout-prunability.git'
if not os.path.exists('/kaggle/working/code'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/code'], check=True)
os.chdir('/kaggle/working/code')
!git pull -q; nvidia-smi --query-gpu=name,memory.total --format=csv; python -c "import torch;print(torch.__version__, torch.cuda.device_count())"
# Use an attached Kaggle dataset copy of CIFAR-10 if present (the Toronto server downloads at ~75 kB/s, 35 min)
os.makedirs('data', exist_ok=True)
tars = glob.glob('/kaggle/input/**/cifar-10-python.tar.gz', recursive=True)
if tars and not os.path.exists('data/cifar-10-python.tar.gz'):
    shutil.copy(tars[0], 'data/cifar-10-python.tar.gz'); print('using dataset copy:', tars[0])
else:
    print('no attached CIFAR-10 tarball found; torchvision will download it (slow)' if not tars else 'tarball already present')
# Resume: reuse trained models from a previous version of this notebook attached as Input (Add Input -> Your Work)
os.makedirs('runs', exist_ok=True); n = 0
for mp in glob.glob('/kaggle/input/**/runs/p*_s*/model.pt', recursive=True):
    d = os.path.basename(os.path.dirname(mp)); dst = os.path.join('runs', d)
    if os.path.exists(os.path.join(dst, 'model.pt')): continue
    os.makedirs(dst, exist_ok=True)
    for f in [os.path.basename(x) for x in glob.glob(os.path.join(os.path.dirname(mp), '*'))]:  # model.pt + all result json
        shutil.copy(os.path.join(os.path.dirname(mp), f), dst)
    n += 1
print(f'restored {n} trained models:', sorted(os.listdir('runs')))

## Config — which runs this session should execute

In [ ]:
EPOCHS = 100
# ---- Dense training this session: one queue per GPU. Done so far: head seeds 0-2 (all p), block seed 0.
# Pass 5 (final report): block dropout seeds 1 and 2. The p=0.0 block model is identical to the head p=0.0
# model (Dropout2d(0) is an Identity), so it is copied from runs/p0.0_s{seed} below instead of retrained.
TRAIN_QUEUES = [
    dict(where='block', ps='0.1 0.3 0.5', seeds='1'),   # GPU 0: 3 runs x ~25 min + sweeps
    dict(where='block', ps='0.1 0.3 0.5', seeds='2'),   # GPU 1
]
# ---- Fine-tuning groups (E1), each job = 20 fine-tune epochs (~5 min). Existing ft_*.json files are skipped.
#   where     : 'head' or 'block' models
#   seeds     : which seeds
#   ps        : dropout rates (None = all)
#   sparsities: targets
#   schedules : 'oneshot' | 'iterative' (4x5 ep) | 'frontload' (2+2+2+14 ep)
FT_GROUPS = [
    # Does block dropout's loss of prunability survive fine-tuning? One-shot, all 3 seeds. 27 jobs
    dict(where='block', seeds=['0', '1', '2'], ps=[0.1, 0.3, 0.5], sparsities=['0.8', '0.9', '0.95'], schedules=['oneshot']),
    # Control for the above at 80% on seeds 1-2 (90/95 are covered by the next group). 2 jobs
    dict(where='head', seeds=['1', '2'], ps=[0.0], sparsities=['0.8'], schedules=['oneshot']),
    # H3 with error bars: head seeds 1-2 at the sparsities where the schedules separate. 48 jobs
    dict(where='head', seeds=['1', '2'], ps=None, sparsities=['0.9', '0.95', '0.98'], schedules=['oneshot', 'iterative']),
    # Front-loaded iterative schedule (most of the budget on the final step), seed 0. 8 jobs
    dict(where='head', seeds=['0'], ps=None, sparsities=['0.95', '0.98'], schedules=['frontload']),
]
# Rough budget: training 6 x 25 min + 6 sweeps ~ 3.2 GPU-h; 85 ft jobs x ~5.3 min ~ 7.5 GPU-h; ~5.5 h wall on 2 GPUs.
os.makedirs('runs', exist_ok=True)


## Pilot (B3): smoke-test the pipeline for 1 epoch before committing hours

In [ ]:
if not glob.glob('runs/p*_s*/model.pt'):
    !python train.py --p 0.0 --seed 0 --epochs 1 --out runs_pilot && python prune_sweep.py --run runs_pilot/p0.0_s0 --skip_layerwise
else: print('models already present, skipping pilot')

## Dense training matrix + pruning sweeps (both GPUs in parallel)

In [ ]:
import torch, subprocess, time, json
ngpu = torch.cuda.device_count()
# 1) pruning sweeps for restored models that lack one (fast, ~5 min each)
restored = [d for d in sorted(glob.glob('runs/p*_s*')) if os.path.exists(f'{d}/model.pt') and not os.path.exists(f'{d}/prune_sweep.json')]
for i in range(0, len(restored), max(ngpu,1)):
    ps = [subprocess.Popen(['python', 'prune_sweep.py', '--run', d], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)))
          for g, d in enumerate(restored[i:i+max(ngpu,1)])]
    for p in ps: p.wait()
# 2) dense training queues, one per GPU (run_matrix.sh skips finished runs and adds the pruning sweep)
procs = []
for g, q in enumerate(TRAIN_QUEUES):
    gpu = str(g % max(ngpu, 1))
    if ngpu <= 1 and procs: procs[-1].wait()   # single GPU: run the queues one after another
    procs.append(subprocess.Popen(['bash', 'run_matrix.sh', q['ps'], q['seeds'], str(EPOCHS), gpu], env=dict(os.environ, WHERE=q['where'])))
for p in procs: p.wait()
!tail -n 1 runs/train_*.log

# 3) p=0.0 block control = head p=0.0 model (no dropout parameters either way): copy instead of retraining,
#    so plots.py --where block has a control for every seed. Fine-tune results are synced both ways.
def sync_block_control():
    for src in sorted(glob.glob('runs/p0.0_s*')):
        seed = src.split('_s')[-1]; dst = f'runs/p0.0blk_s{seed}'
        if not os.path.exists(f'{src}/model.pt'): continue
        os.makedirs(dst, exist_ok=True)
        if not os.path.exists(f'{dst}/model.pt'):
            shutil.copy(f'{src}/model.pt', dst)
            for f in ['train_log.json', 'prune_sweep.json']:
                if os.path.exists(f'{src}/{f}'):
                    j = json.load(open(f'{src}/{f}'))
                    if f == 'train_log.json': j['args']['where'] = 'block'
                    else: j['where'] = 'block'
                    json.dump(j, open(f'{dst}/{f}', 'w'))
        for a, b in [(src, dst), (dst, src)]:
            for f in glob.glob(f'{a}/ft_*.json'):
                if not os.path.exists(f'{b}/{os.path.basename(f)}'): shutil.copy(f, b)
sync_block_control()
print(sorted(os.listdir('runs')))


## E1 fine-tuning groups (FT_GROUPS), both GPUs in parallel


In [ ]:
def ft_jobs():
    jobs = []
    for g in FT_GROUPS:
        suf = 'blk' if g['where'] == 'block' else ''
        for r in sorted(glob.glob(f'runs/p*{suf}_s*')):
            if ('blk' in r) != (g['where'] == 'block'): continue
            if not os.path.exists(f'{r}/model.pt'): continue
            p = float(os.path.basename(r).split('_s')[0].lstrip('p').replace('blk', ''))
            seed = r.split('_s')[-1]
            if seed not in g['seeds'] or (g['ps'] is not None and p not in g['ps']): continue
            for s in g['sparsities']:
                for sch in g['schedules']:
                    if not os.path.exists(f'{r}/ft_{sch}_s{s}.json') and (r, s, sch) not in jobs: jobs.append((r, s, sch))
    return jobs
jobs = ft_jobs()
print(len(jobs), 'fine-tuning jobs:', jobs)
queues = [jobs[i::max(ngpu,1)] for i in range(max(ngpu,1))]
def run_queue(q, gpu):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    cmd = ' ; '.join(f'python finetune.py --run {r} --sparsity {s} --schedule {sch}' for r, s, sch in q) or 'true'
    return subprocess.Popen(['bash', '-c', cmd], env=env)
ps = [run_queue(q, g) for g, q in enumerate(queues)]
for p in ps: p.wait()
sync_block_control()
print(len(ft_jobs()), 'fine-tuning jobs still missing')


## Figures + tables

In [ ]:
!pip -q install tabulate
!python plots.py --runs runs --out figures --where head
!python plots.py --runs runs --out figures_block --where block
!ls figures
!cd /kaggle/working && tar czf results_$(date +%Y%m%d_%H%M).tar.gz code/runs/*/*.json code/runs/*.log code/figures* && ls -la /kaggle/working/*.tar.gz
from IPython.display import Image, display
for f in ['acc_vs_sparsity.png', 'acc_vs_sparsity_bnrecal.png', 'finetune.png', 'weight_hist.png']:
    if os.path.exists('figures/' + f): display(Image('figures/' + f))